In [2]:
import logging
import os
from tqdm import tqdm
import SimpleITK as sitk
import numpy as np
import sys
from pathlib import Path
from random import randint

log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)

MRI_FOLDER = "output/extract1/images/"
# ANNOTATION_FOLDER = "output/extract1/labels/"
OUTPUT_DIR = "output/std1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def setup_logger():
    logger = logging.getLogger(__name__)
    
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
        handler.close()
    
    logger.setLevel(logging.DEBUG)
    
    logger.propagate = False
    log_file = os.path.join(log_dir, 'logs_std.log')
    
    # Add file handler
    file_handler = logging.FileHandler(log_file)
    file_handler.setLevel(logging.DEBUG)
    
    # Add console handler
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO) 
    
    # Create formatter
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)
    
    # Add handlers
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    
    return logger

# Initialize logger
logger = setup_logger()

NEW_SIZE = [224, 224, 50]

logger.info(f"Starting parameter logging")

logger.debug("Debug logging is enabled")



2025-07-18 15:26:17,761 - INFO - Starting parameter logging


In [ ]:
def resize_image_itk(sitk_image, new_size, resample_method=sitk.sitkNearestNeighbor):
    """
    Resize a SimpleITK image to a new size.
    
    Parameters:
    sitk_image (sitk.Image): The input image
    new_size (list or tuple): Target size (should be integers)
    resample_method (int): SimpleITK interpolation method
    
    Returns:
    sitk.Image: Resampled image
    """

    new_size = [int(s) for s in new_size]

    original_size = sitk_image.GetSize()
    logger.info(f"original size is {original_size}")
    original_spacing = sitk_image.GetSpacing()

    dim = sitk_image.GetDimension()
    new_spacing = [original_spacing[i] * (original_size[i] / new_size[i]) for i in range(dim)]

    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(new_size)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetOutputOrigin(sitk_image.GetOrigin())
    resampler.SetOutputDirection(sitk_image.GetDirection())
    resampler.SetInterpolator(resample_method)
    resampler.SetDefaultPixelValue(0)

    resized_image = resampler.Execute(sitk_image)
    
    return resized_image

In [ ]:
if __name__ == "__main__":
    mri_folder = MRI_FOLDER

    output_dir = OUTPUT_DIR
    os.makedirs(output_dir, exist_ok=True)

    mri_paths = [os.path.join(mri_folder, f) for f in os.listdir(mri_folder) 
                if f.endswith('.nii.gz')]
    
    for mri_path in tqdm(mri_paths, desc = "Processing mri_files", unit="file"):
        mri_image = None
        logger.info(f"............Starting process for {mri_path}")
        try:
            logger.info(f"Loading MRI image from {mri_path}")
            mri_image = sitk.ReadImage(mri_path)
            
            resized_image = resize_image_itk(mri_image, NEW_SIZE)

            logger.info(f"new size is {resized_image.GetSpacing()}")
            
            output_filename = f"res_{mri_path}"
            output_path = os.path.join(OUTPUT_DIR, output_filename)
            logger.info(f"saved file with filename {output_filename}")

            sitk.WriteImage(resized_image, output_path)


        except Exception as e:
            logger.error(f"Error processing {mri_path}: {str(e)}")
            continue